<a href="https://colab.research.google.com/github/0zzge/traffic-sign-recognition-ml/blob/main/traffic_sign_recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install rarfile
from google.colab import files
import rarfile
import os
from PIL import Image
from pathlib import Path

# Step 1: Upload the File
print("Please select your ... .rar file.")
uploaded = files.upload()

# Step 2: Get the name of the uploaded file dynamically.
if not uploaded:
    print("ERROR: No files were uploaded.")
    # Stop code on error
    raise ValueError("The file upload was cancelled or failed.")

# Get the name of the first file uploaded.
rar_name = list(uploaded.keys())[0]
print(f"'{rar_name}' The file has been uploaded successfully.")

# Check if the uploaded file is a .rar file.
if not rar_name.lower().endswith('.rar'):
    print(f"ERROR: The file you uploaded, '{rar_name}', is not a RAR file.")
    raise ValueError("Please upload a file with the .rar extension.")

#Step 3: Open the RAR file and extract its contents to a folder.
extract_folder = "extracted_images"  # The folder where the images will be extracted
os.makedirs(extract_folder, exist_ok=True)

try:
    with rarfile.RarFile(rar_name, 'r') as rar_ref:
        rar_ref.extractall(extract_folder)
    print(f"The contents of the file '{rar_name}' have been successfully extracted to the folder '{extract_folder}'.")
except rarfile.BadRarFile:
    print(f"ERROR: '{rar_name}' is not a valid RAR file or is corrupted.")
    raise
except Exception as e:
    print(f"An error occurred while extracting the RAR file: {e}")
    raise

#Step 4: Converting Images to PNG
input_folder = extract_folder
output_folder = "png_images"  # The folder where PNG files will be saved.
os.makedirs(output_folder, exist_ok=True)

input_path = Path(input_folder)

# Search for both .jpg and .jpeg files in ALL subfolders.
image_files = list(input_path.rglob("*.jpg")) + list(input_path.rglob("*.jpeg"))

if not image_files:
    print(f"Warning: No .jpg or .jpeg images were found in the '{input_folder}' folder (or its subfolders).")
else:
    print(f"{len(image_files)} images found, conversion is starting...")

# Step 5: Converting and Saving
count = 0
for img_path in image_files:
    try:
        img = Image.open(img_path)

        # Create a new file name (original name + .png)
        new_filename = img_path.stem + ".png"
        output_path = os.path.join(output_folder, new_filename)

        img.save(output_path)
        count += 1
    except Exception as e:
        print(f"Error: Could not process file {img_path} - {e}")

if count > 0:
    print(f"The process is complete. {count} images have been saved as PNG files to the '{output_folder}' folder.")
else:
    print("The process was completed because no image to convert was found.")


Please select your ... .rar file.


In [ ]:
import pandas as pd
from PIL import Image
from pathlib import Path
import os

# 1. The main folder containing the PNG files.
input_folder = "png_images"

# 2. An empty list for the data to be collected in the DataFrame.
dataframe_data = []

# 3. Check if the folder exists.
if not os.path.exists(input_folder):
    print(f"ERROR: A folder named '{input_folder}' could not be found.")
    print("Please make sure you have run the previous step (extracting and converting from RAR).")
else:
    print(f"The folder '{input_folder}' was found successfully.")

    # Create Path object
    input_path = Path(input_folder)

    # 4. Find all .png files in the folder and (if any) subfolders.
    image_files = list(input_path.rglob("*.png"))

    if not image_files:
        print(f"Warning: No .png images were found in the '{input_folder}' folder.")
    else:
        print(f"{len(image_files)} .png files were found. Information is being gathered...")

        # 5. Loop through each image file.
        for img_path in image_files:
            try:
                # Open image
                img = Image.open(img_path)

                # Get its information (width, height, color mode)
                width, height = img.size
                mode = img.mode

                # 6. Add the collected information to the list as a dictionary.
                dataframe_data.append({
                    'file_path': str(img_path),
                    'width': width,
                    'height': height,
                    'color_mode': mode
                })
            except Exception as e:
                # The error message will appear if the image file is corrupted or cannot be opened.
                print(f"Error: Could not read file {img_path} - {e}")

        # 7. Create the DataFrame after the loop is complete.
        if dataframe_data:
            df = pd.DataFrame(dataframe_data)

            print("\nThe generated Image DataFrame:")
            print(df)

# Optional: If you want to save the DataFrame as a CSV file
# df.to_csv("png_image_info.csv", index=False)
# print("\nDataFrame saved as 'png_image_info.csv'.")
        else:
            print("The dataframe could not be created (no valid images were found to add to the list).")

In [ ]:
# Let's install the necessary libraries
# OpenCV is a powerful tool for variance and corrupted file detection.
!pip install opencv-python-headless pillow numpy tqdm

import os
import cv2  # OpenCV library
import numpy as np
from PIL import Image
from pathlib import Path
from tqdm import tqdm # progress bar

print("The process of cleaning up noisy and distorted images is beginning...")

# STEP 1: Settings and Folder Definition
image_folder = "png_images" # The folder containing the PNGs to be checked (created in the previous step)
min_filesize_kb = 2         # Minimum file size (files smaller than 2 KB should be deleted)
min_dimension = 30         # Minimum width/height (remove anything smaller than 50 pixels)
low_variance_threshold = 5 # Low variance threshold (values ​​below this are considered "empty")

input_path = Path(image_folder)
try:
    image_files = list(input_path.glob("*.png"))
except FileNotFoundError:
    print(f"ERROR: Folder '{image_folder}' could not be found.")
    print("Please make sure you run the step to convert the images to .png format first.")
    # Stop code in case of error
    raise

if not image_files:
    print(f"ERROR: No .png images were found in the '{image_folder}' folder.")
else:
    print(f"{len(image_files)} images were found. Scanning is starting...")

    deleted_count = 0
    corruption_count = 0
    filesize_count = 0
    dimension_count = 0
    variance_count = 0

    # STEP 2: Image Review Cycle
    for img_path in tqdm(image_files, desc="Resimler Taranıyor"):
        try: # Outer try for all checks on an image
            # --------
            #Criterion 1: Corrupted File Check (Files that cannot be opened)
            # --------
            try: # Inner try for initial image opening
                # First, perform a quick verification with PIL.
                img = Image.open(img_path)
                img.verify() # Verify file integrity
                img.close() # Close to reopen the file

                # Now read the file with OpenCV (for more detailed checking and variance)
                # OpenCV doesn't like paths with Turkish characters, so use str()
                img_cv = cv2.imread(str(img_path))

                if img_cv is None: # If OpenCV cannot read the file (returns None)
                    raise Exception("OpenCV could not read the file (it's probably corrupted).")

            except Exception as e: # Except for inner try (Kriter 1)
                # print(f"\n[CORRUPT] {img_path.name} is corrupt or unreadable. Error: {e}")
                os.remove(img_path) # delete file
                deleted_count += 1
                corruption_count += 1
                continue # Move to the next file

            # --------
            # Criterion 2: File Size Control
            # --------
            filesize = os.path.getsize(img_path) / 1024 # KB cinsinden
            if filesize < min_filesize_kb:
                #print(f"\n[SMALL FILE] {img_path.name} is too small ({filesize:.2f} KB). It is being deleted...")
                os.remove(img_path)
                deleted_count += 1
                filesize_count += 1
                continue

            # --------
            # Criterion 3: Image Dimensions Check
            # --------
            # OpenCV gives the following values ​​in order: height, width, channels.
            height, width, _ = img_cv.shape
            if width < min_dimension or height < min_dimension:
                # print(f"\n[SMALL SIZE] {img_path.name} is too small ({width}x{height}). It is being deleted...")
                os.remove(img_path)
                deleted_count += 1
                dimension_count += 1
                continue

            # --------
            # Criterion 4: Low Variance Check
            # --------
            # Check if the image is in color (to create a gray_img).
            if len(img_cv.shape) == 3: # Color image (RGB/BGR)
                gray_img = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)
            else: # Already grayscale
                gray_img = img_cv

            variance = np.var(gray_img)
            if variance < low_variance_threshold:
                # print(f"\n[LOW VARIANCE] {img_path.name} is being deleted with low variance ({variance:.2f}).")
                os.remove(img_path)
                deleted_count += 1
                variance_count += 1
                continue

        except Exception as e: # Outer except for any other errors during image processing in the loop
            print(f"\n[GENERAL ERROR] An unexpected error occurred while processing {img_path.name}: {e}")

            if os.path.exists(img_path):
                os.remove(img_path)
            deleted_count += 1
            continue

# STEP 3: Report
print("\n--- Cleaning Process Completed ---")
print(f"A total of {deleted_count} images were deleted.")
print(f" - {corruption_count} corrupted/unreadable files.")
print(f" - {filesize_count} very small files (Limit: {min_filesize_kb} KB).")
print(f" - {dimension_count} very small images (Limit: {min_dimension}x{min_dimension} px).")
print(f" - {variance_count} low variance (empty/frozen) images (Limit: {low_variance_threshold}).")
print(f"Number of remaining healthy images: {len(image_files) - deleted_count}")
print("\nYou can now run the normalization step (which creates normalized_npy).")

In [ ]:
# Install the necessary libraries (they are usually installed in Colab, but just to be sure)
!pip install numpy pillow tqdm

import os
import numpy as np
from PIL import Image
from pathlib import Path
from tqdm import tqdm # To view transaction progress

# STEP 1: Defining Folders
# This folder was already created in your previous step.
input_folder = "png_images"
# New folder where normalized .npy files will be saved
output_folder = "normalized_npy2"

# Create the output folder
os.makedirs(output_folder, exist_ok=True)

# STEP 2: Finding PNG Files
input_path = Path(input_folder)
image_files = list(input_path.glob("*.png"))

if not image_files:
    print(f"ERROR: No .png image found in folder '{input_folder}'.")
else:
    print(f"{len(image_files)} PNG images were found. Normalization is starting...")

    # STEP 3: Normalization Cycle
    count = 0
    #With tqdm you'll see a nice progress bar in Colab.
    for img_path in tqdm(image_files, desc="Images are being normalized."):
        try:
            # Open the image and convert it to RGB format (to be sure)
            img = Image.open(img_path).convert('RGB')

            # Translate to official NumPy sequence
            img_array = np.array(img)

          # Normalization:
          # 1. Convert type to float32 (decimal)
          # 2. Scale values ​​to the range [0.0, 1.0] by dividing by 255.0
            normalized_array = img_array.astype(np.float32) / 255.0

            # Create a new file name (e.g., image.png -> image.npy)
            new_filename = img_path.stem + ".npy"
            output_path = os.path.join(output_folder, new_filename)


            np.save(output_path, normalized_array)
            count += 1

        except Exception as e:
            print(f"Error: Could not process file {img_path} - {e}")

    if count > 0:
        print(f"\nTransaction completed!")
        print(f"{count} images were normalized and saved as .npy files in the '{output_folder}' folder.")
    else:
        print("No images were found to process.")

In [ ]:
import os
import random
from pathlib import Path

# STEP 1: Folder Definition
npy_folder = "normalized_npy2" # The folder containing your normalized .npy files

# STEP 2: Gathering the Paths of All .npy Files
try:
# Using Pathlib, we create a list containing the full file paths (or: 'normalized npy/file1.npy').
    npy_path = Path(npy_folder)
# Use .glob() to retrieve only files with the .npy extension
# Use list() to convert this object to a list
    all_npy_files = list(npy_path.glob("*.npy"))

   # Convert file paths to strings (more compatible with some libraries)
    all_npy_files = [str(path) for path in all_npy_files]

    if not all_npy_files:
        print(f"ERROR: No .npy file found in folder '{npy_folder}'.")
    else:
        print(f"A total of {len(all_npy_files)} .npy files were found in the '{npy_folder}' folder.")
        print("\n--- Before Mixing (First 5 Files) ---")
        # Print the first 5 files in sequential order (usually alphabetical)
        print("\n".join(sorted(all_npy_files)[:5]))

        # STEP 3: Shuffle the List
        # random.shuffle() shuffles the list "in-place".
        # You don't need to assign it to a new variable.
        random.shuffle(all_npy_files)

        # STEP 4: Displaying the Result
        print("\n" + "---" * 10 + "\n")
        print("The file list has been successfully shuffled!")
        print("--- After Mixing (First 5 Files) ---")
        print("\n".join(all_npy_files[:5]))

        print(f"\nYou can now use the shuffled list 'all_npy_files' with elements '{len(all_npy_files)}' for training purposes.")

except FileNotFoundError:
    print(f"ERROR: Folder '{npy_folder}' not found.")
    print("Please make sure you run the normalization step first.")
except Exception as e:
    print(f"Please make sure you run the normalization step first.")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from pathlib import Path
from tqdm import tqdm

# --- STEP 1: LOADING DATA (X) AND LABELS (y) ---
# This step should use the 'all_npy_files' list in your code and
# a (hypothetical) CSV file containing your labels.

# --- THIS PART IS JUST A SIMULATION (EXAMPLE) ---
# You already have your 'all_npy_files' list (from your last code)
# Let's assume you have 9300 file paths as an example:
# all_npy_files = [...] # Your list with 9300 elements

# HOW TO GET THE LABELS?
# Assumption: Let's assume you have a file named 'labels.csv' and
# this file has columns 'file_name' and 'class_id'.
# (If your tags are in a different location, you need to change this part)

# ----- CREATING SAMPLE DATA (To run the code) -----

# Replace this part with your own data load in your actual code.
print("Simulation: Loading (assuming) real data and labels...")
N_SAMPLES = 10000
K_SINIFLAR = 38 # Traffic signs generally have 43 classes.

# 1. Let's create fake X data (You will load the .npy files)
# In reality, you will do it like this:
# X_data = np.array([np.load(f) for f in tqdm(all_npy_files, desc="Loading NPY Files")])
X_data = np.random.rand(N_SAMPLES, 32, 32, 3) # 9300 fake 32x32x3 pictures

# 2. Let's create fake y labels (You will pull them from the CSV or file name)
# In reality, you would do it like this (assuming CSV):
# df_labels = pd.read_csv("labels.csv")
# y_data = df_labels['class_id'].values
y_data = np.random.randint(0, K_SINIFLAR, N_SAMPLES) # 9300 tane sahte etiket

print(f"Data (X) loaded. Shape: {X_data.shape}")
print(f"Labels (y) loaded. Shape: {y_data.shape}")
print("--- Simulation End ---")
# ----- END: SAMPLE DATA CREATION -----


# --- STEP 2: K-FOLD CROSS-VALIDATION IMPLEMENTATION ---

# Define your model here (e.g., a CNN with TensorFlow/Keras)
# Example:
# def create_model():
# model = Sequential([...])
# model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
# return model

# K-Fold settings
K = 5 # Divide the data into 5 parts (Usually 5 or 10 are used)
# shuffle=True shuffles the data before dividing it. # (Your 'all_npy_files' list was already shuffled, but it's good practice for KFold to
# shuffle X and Y again)
kf = KFold(n_splits=K, shuffle=True, random_state=42)

print(f"\n--- Initiating {K}-Fold Cross-Validation ---")

# A list to store performance results.
fold_scores = []
fold_no = 1

# The kf.split() function generates pairs (train_index, test_index) using X_data.
# We split both X and y using these indices.
for train_index, test_index in kf.split(X_data, y_data):

    print(f"\n=============== FOLD {fold_no}/{K} ===============")

    # 1. Allocate the data for this fold.
    X_train, X_test = X_data[train_index], X_data[test_index]
    y_train, y_test = y_data[train_index], y_data[test_index]

    print(f"   Train data count: {len(X_train)}")
    print(f"   Test (Validation) data count: {len(X_test)}")

    # 2. Create and compile the model (Place your model here)
    # model = create_model()
    print("   Model oluşturuldu ve derlendi (Simülasyon)...")

    # 3. Train the model
    print("   The model is being trained...")
    # history = model.fit(X_train, y_train,
    #                     epochs=10,
    #                     batch_size=32,
    #                     validation_data=(X_test, y_test),
    #                     verbose=0) # verbose=0 logları kapatır
    print("   Model training completed.")

    #4. Evaluate the model with this fold's test data.
    print("   The model is being evaluated...")
    # loss, accuracy = model.evaluate(X_test, y_test, verbose=0)

    # A random performance for the simulation.
    accuracy = 0.85 + (random.random() * 0.1)

    print(f"   Fold {fold_no} Test Performance (Accuracy): {accuracy * 100:.2f}%")
    fold_scores.append(accuracy)

    fold_no += 1

# --- STEP 3: SHOW RESULTS ---
print("\n" + "="*30)
print("All Folds Completed!")
print(f"{K}-Fold Cross-Validation Average Performance: {np.mean(fold_scores) * 100:.2f}%")
print(f"Standard Deviation of Performance: {np.std(fold_scores) * 100:.2f}%")
print("A high average performance and a low standard deviation are desired.")

In [ ]:
import numpy as np
import pandas as pd
import random
from pathlib import Path
from sklearn.model_selection import train_test_split
import os
import shutil
# Required libraries for Keras' Data Generator
from tensorflow.keras.preprocessing.image import ImageDataGenerator


## --- STEP 1: Preparing Settings and Folders ---
RAW_IMAGE_FOLDER = "png_images" # The folder containing PNG files
TARGET_SIZE = (64, 64)        # Reduced size for CNN
TOTAL_EXPECTED_CLASSES = 38   # Number of classes (43 for Traffic Signs)
TEST_SIZE_RATE = 0.2          # 20% of the total data will be the test set.

# Create folders for the training and validation kit
base_dir = Path("resized_dataset_64x64")
train_dir = base_dir / "train"
test_dir = base_dir / "test"
os.makedirs(train_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)
print(f"Target size: {TARGET_SIZE}. Data will be loaded in a RAM-friendly manner.")


## --- STEP 2: Extracting Labels and Creating a DataFrame ---

# We collect the paths and labels of all PNG files.
all_image_files = list(Path(RAW_IMAGE_FOLDER).rglob("*.png"))
dataframe_data = []

for file_path in all_image_files:
    # tag is removed from the first part of the filename (e.g., '00001_...' -> '00001')
    class_label = file_path.name.split('_')[0]

    # Invalid tag check (if not cleaned in previous steps)
    if class_label.isdigit() and 0 <= int(class_label) < TOTAL_EXPECTED_CLASSES:
        dataframe_data.append({
            'filename': file_path.name,
            'class_id': class_label,
            'filepath': str(file_path)# We are keeping the original file path.

        })

df = pd.DataFrame(dataframe_data)
df['class_id'] = df['class_id'].astype(str) # Keras's desired format

# Check the number of files with incorrect/invalid tags.
invalid_count = len(all_image_files) - len(df)
print(f"\n{invalid_count} invalid or unfiltered files were skipped.")
print(f"A total of {len(df)} valid files were found.")


## --- STEP 3: Stratified Data Partitioning (Train/Test) ---

# Check and remove classes that have only one instance.
class_counts = df['class_id'].value_counts()
single_sample_classes = class_counts[class_counts == 1].index.tolist()

if single_sample_classes:
    print(f"\nWarning: Single-sample classes were found and are being removed from the dataset: {single_sample_classes}")
    df = df[~df['class_id'].isin(single_sample_classes)]
    print(f"After removing single instance classes, the valid file {len(df)} remains.")

# We divide the dataset into training and final test sets.
# We maintain class ratios in both sets using Stratify.
X_train_info, X_test_info = train_test_split(
    df,
    test_size=TEST_SIZE_RATE,
    random_state=42,
    stratify=df['class_id']
)

print("\n--- Data Splitting Results ---")

print(f"TRAIN Set: {len(X_train_info)} images")
print(f"TEST Set: {len(X_test_info)} images")

# At this stage, we have the X_train_info and X_test_info DataFrames.
# For Keras' Data Generator to work easily,
# we need to copy these images to the training and test folders.
# Note: This does not fully load RAM, it only copies the PNG files
# to the appropriate folder structure.

def move_files(dataframe, target_dir):
    for index, row in dataframe.iterrows():
        source = Path(row['filepath'])
        destination = target_dir / row['class_id'] / source.name
        os.makedirs(destination.parent, exist_ok=True)
        shutil.copy(source, destination)

print("Copying files from the training set...")
move_files(X_train_info, train_dir)
print("Copying files from the test set...")
move_files(X_test_info, test_dir)


## --- STEP 4: Creating Keras Data Generators ---

# Generators that will read, resize, and normalize images from disk
# This section allows you to read from disk and perform preprocessing (normalization, resizing) while using the data, instead of just copying the data to disk.
# This allows you to read from disk and perform preprocessing (normalization, resizing) while using the data.

# Data augmentation can only be done for training data.
train_datagen = ImageDataGenerator(
    rescale=1./255, # Normalization (scaling to the 0-1 range)
    validation_split=0.1875 # 18.75% of the training set will be the validation set (out of 15% of the total).
)

test_datagen = ImageDataGenerator(rescale=1./255) # Test data is simply normalized.

# Education data generator
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=TARGET_SIZE,
    batch_size=32, # Example batch size
    class_mode='categorical',
    subset='training',
    seed=42
)

# Validation data generator
validation_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=TARGET_SIZE,
    batch_size=32,
    class_mode='categorical',
    subset='validation',
    seed=42
)

# Test data generator
test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=TARGET_SIZE,
    batch_size=32,
    class_mode='categorical',
    shuffle=False, # We do not mix anything for the test set.
    seed=42
)

print(f"\nData generators are ready. Training, validation, and testing data are now available in a RAM-friendly format.")
# This step should be used directly in your next CNN model training cell.

In [ ]:
import numpy as np
import pandas as pd
import random
from pathlib import Path
from tqdm import tqdm
from PIL import Image # Required for resizing

# --- Settings (Taken from the previous cell) ---
NPY_FOLDER = "normalized_npy2"
TARGET_SIZE = (64, 64) # Target reduced size
TOTAL_EXPECTED_CLASSES = 38

# --------------------------
# STEP 2: Uploading NPY Files and Removing Labels (RAM-Friendly)
# --------------------------

npy_path = Path(NPY_FOLDER)
all_npy_files = list(npy_path.glob("*.npy"))
all_npy_files = [str(path) for path in all_npy_files]
random.shuffle(all_npy_files)

if not all_npy_files:
    raise FileNotFoundError(f"ERROR: The .npy file was not found in folder '{NPY_FOLDER}'. Please check the previous steps.")

print(f"A total of {len(all_npy_files)} .npy files were found.")

print(f"Images will be resized to {TARGET_SIZE} during loading...")

X_list_resized = [] # Corrected list name
y_labels_list = []
invalid_file_count = 0 # Invalid tag counter

for file_path in tqdm(all_npy_files, desc="Data is being loaded and sized."):
    try:
        # Remove the tag from the filename and check.
        class_label_str = Path(file_path).name.split('_')[0]

        if not class_label_str.isdigit():
            invalid_file_count += 1
            continue

        # 1. Upload the large (416x416) .npy file.
        img_array_large = np.load(file_path)

        # 2. Convert to PIL Image for resizing (revert to 0-255 range)
        img = Image.fromarray((img_array_large * 255).astype(np.uint8))

        # 3. Resize (to 64x64)
        # Image.Resampling.LANCZOS gives better results for reduction.
        img_resized = img.resize(TARGET_SIZE, Image.Resampling.LANCZOS)

        # 4. Normalize again to the 0-1 range and add to the list.
        X_list_resized.append(np.array(img_resized).astype(np.float32) / 255.0)

        # We add the tag ONLY if it's valid.
        y_labels_list.append(class_label_str)

    except Exception as e:
        print(f"Warning: {Path(file_path).name} could not be loaded. Error: {e}")
# --- Create array X (now smaller size) ---
X = np.array(X_list_resized)
y_labels = np.array(y_labels_list)

# Convert labels to integers and check if they are within the valid range (0-42 for GTSRB)
y_int_0_indexed = y_labels.astype(int)
valid_indices = (y_int_0_indexed >= 0) & (y_int_0_indexed < TOTAL_EXPECTED_CLASSES)

if not np.all(valid_indices):
    print(f"Warning: {np.sum(~valid_indices)} images with invalid tags (e.g., class > 42) were found and eliminated.")
    X = X[valid_indices]
    y_int_0_indexed = y_int_0_indexed[valid_indices]

print(f"\nLoading and sizing complete.")

print(f"X (Images) Shape: {X.shape}, y (Labels) Shape: {y_int_0_indexed.shape}")

In [ ]:
import numpy as np
import pandas as pd
import random
from pathlib import Path
from tqdm import tqdm
from PIL import Image
from tensorflow.keras.utils import to_categorical
import os

# --- Settings (from Cell 13) ---
NPY_FOLDER = "normalized_npy2"
TARGET_SIZE = (64, 64) # We will reduce the images to this size.
TOTAL_EXPECTED_CLASSES = 38 # Taken from cell 19.

# Locate the .npy files (from cell 14)
npy_path = Path(NPY_FOLDER)
all_npy_files = list(npy_path.glob("*.npy"))
all_npy_files = [str(path) for path in all_npy_files]
random.shuffle(all_npy_files)

print(f"A total of {len(all_npy_files)} .npy files were found.")
print(f"Images will be resized to {TARGET_SIZE} during loading...")

X_list_resized = []
y_labels_list = []
invalid_file_count = 0 # To count files with incorrect tags

# --- Combination of Cells 14 and 19 (RAM-Friendly Installation) ---
for file_path in tqdm(all_npy_files, desc="Data is being loaded and sized."):
    try:
        # 5. Remove the label from the filename (Logic in cell 14)
        class_label_str = Path(file_path).name.split('_')[0]

        # --- NEW CHECK ---
        # Check if the label consists entirely of numbers.
        if not class_label_str.isdigit():
            invalid_file_count += 1
            continue # This file does not have a numerical label, skip it.
        # --- END OF CHECK ---

        # 1. Upload the large (416x416) .npy file
        img_array_large = np.load(file_path)

        # 2. Convert to PIL Image for resizing (revert to 0-255 range)
        img = Image.fromarray((img_array_large * 255).astype(np.uint8))

        # 3. Resize (TARGET_SIZE = 64x64)
        img_resized = img.resize(TARGET_SIZE, Image.Resampling.LANCZOS)

        # 4. Normalize again to the 0-1 range and add to the list (We are adding the small array)
        # We add image data ONLY if the label is valid
        X_list_resized.append(np.array(img_resized).astype(np.float32) / 255.0)

        # We add the tag ONLY if the tag is valid.
        y_labels_list.append(class_label_str)

    except Exception as e:
        print(f"Warning: {Path(file_path).name} could not be loaded. Error: {e}")

# --- Create array X (now smaller size) ---
X = np.array(X_list_resized)
y_labels = np.array(y_labels_list)

print(f"\n{invalid_file_count} files with invalid labels like 'road74' were skipped.")
print(f"Loading and sizing completed.")
print(f"X (Images) Shape: {X.shape}, y (Labels) Shape: {y_labels.shape}")

# --- Label Transformation and Filtering (From Cell 19) ---

# We are now certain that everything in the y_labels array is a numeric string.
y_int_0_indexed = y_labels.astype(int)

# Filter out invalid tags (less than 0 or greater than 43)
valid_indices = (y_int_0_indexed >= 0) & (y_int_0_indexed < TOTAL_EXPECTED_CLASSES)

if not np.all(valid_indices):
    print(f"Warning: {np.sum(~valid_indices)} images with invalid tags (e.g., class > 42) were found and eliminated.")
    X = X[valid_indices]
    y_int_0_indexed = y_int_0_indexed[valid_indices]

print(f"Shape of X after filtering: {X.shape}")
print(f"Shape of y (integer label) after filtering: {y_int_0_indexed.shape}")

# --- One-Hot Encoding (For Deep Learning, NOT REQUIRED for Decision Tree) ---
# y = to_categorical(y_int_0_indexed, num_classes=TOTAL_EXPECTED_CLASSES)
# print(f"Shape of y (One-Hot Labels): {y.shape}")
print("\nData is now ready for Random Forest and data splitting steps.")
print("Variables to use: X and y_int_0_indexed")

In [ ]:
import numpy as np
import pandas as pd
import random
from pathlib import Path
from tqdm import tqdm
from PIL import Image # CRITICAL: Required for resizing
from tensorflow.keras.utils import to_categorical

# --- Settings ---
NPY_FOLDER = "normalized_npy2"
TARGET_SIZE = (64, 64) # Target reduced size
TOTAL_EXPECTED_CLASSES = 38

# Find .npy files
npy_path = Path(NPY_FOLDER)
all_npy_files = list(npy_path.glob("*.npy"))
all_npy_files = [str(path) for path in all_npy_files]
random.shuffle(all_npy_files)

if not all_npy_files:
    raise FileNotFoundError(f"ERROR: No .npy files found in folder '{NPY_FOLDER}'. Please check previous steps.")

print(f"Total {len(all_npy_files)} .npy files found.")
print(f"Images will be resized to {TARGET_SIZE} during loading...")

X_list_resized = [] # Holds the resized (smaller) arrays
y_labels_list = []
invalid_file_count = 0

# --- CORRECTED AND RAM-FRIENDLY LOADING ---
for file_path in tqdm(all_npy_files, desc="Loading and Resizing Data"):
    try:
        # Extract label from the FIRST PART of the filename
        class_label_str = Path(file_path).name.split('_')[0]

        if not class_label_str.isdigit():
            invalid_file_count += 1
            continue

        # 1. Load the large (416x416) .npy file (This enters RAM only for this loop)
        img_array_large = np.load(file_path)

        # 2. Convert to PIL Image for resizing (revert to 0-255 range)
        img = Image.fromarray((img_array_large * 255).astype(np.uint8))

        # 3. Resize (to 64x64)
        img_resized = img.resize(TARGET_SIZE, Image.Resampling.LANCZOS)

        # 4. Normalize back to 0-1 range and add to list (Only small array is kept)
        X_list_resized.append(np.array(img_resized).astype(np.float32) / 255.0)

        # Add label
        y_labels_list.append(class_label_str)

    except Exception as e:
        print(f"Warning: {Path(file_path).name} could not be loaded. Error: {e}")

# --- Create X array (now smaller size) ---
X = np.array(X_list_resized)
y_labels = np.array(y_labels_list)

print(f"\n{invalid_file_count} files with invalid labels skipped.")
print(f"Loading and resizing complete.")
print(f"X (Images) Shape: {X.shape}, y (Labels) Shape: {y_labels.shape}")

# --- Label Filtering ---
y_int_0_indexed = y_labels.astype(int)
valid_indices = (y_int_0_indexed >= 0) & (y_int_0_indexed < TOTAL_EXPECTED_CLASSES)

if not np.all(valid_indices):
    print(f"Warning: {np.sum(~valid_indices)} images with invalid labels found and removed.")
    X = X[valid_indices]
    y_int_0_indexed = y_int_0_indexed[valid_indices]

print(f"X shape after filtering: {X.shape}")
print(f"y (integer label) shape after filtering: {y_int_0_indexed.shape}")

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

# --- Corrected Split (Replacing Cell 16) ---
# We use X (shape 424, 64, 64, 3) and y_int_0_indexed (integer labels 0-42, shape 424,)
# which were created and filtered in the previous steps.

print(f"X shape for splitting: {X.shape}")
print(f"y (label) shape for splitting: {y_int_0_indexed.shape}")

# Settings (From previous context)
TEST_SIZE_RATE = 0.2  # Reserve 20% for testing (More data ensures better stability)
# Reserve validation set from the remaining data (80%) (e.g., 15% of total)
# (0.15 / (1.0 - 0.20)) = 0.15 / 0.80 = 0.1875
VAL_RATE_ADJUSTED = 0.15 / (1.0 - TEST_SIZE_RATE)

# 1. Split: Create Train+Val and Test sets
# Using stratify=y_int_0_indexed ensures proportional class distribution.
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y_int_0_indexed,
    test_size=TEST_SIZE_RATE,
    random_state=42,
    stratify=y_int_0_indexed
)

# 2. Split: Separate Train and Validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=VAL_RATE_ADJUSTED,
    random_state=42,
    stratify=y_train_val
)

print("\n--- Data Splitting Results ---")
print(f"TRAIN Set (Training): {len(X_train)} images (Shape: {X_train.shape})")
print(f"VAL Set (Validation): {len(X_val)} images (Shape: {X_val.shape})")
print(f"TEST Set (Final Test): {len(X_test)} images (Shape: {X_test.shape})")

print(f"\nTraining set label (y_train) shape: {y_train.shape}")
print(f"Test set label (y_test) shape: {y_test.shape}")

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

# --- CORRECTED SETTINGS ---
TEST_SIZE_RATE = 0.2  # 20% for testing
VAL_SIZE_RATE = 0.15  # 15% for validation (based on total data)
# -----------------------------

# --- Corrected Splitting Logic ---
# y_int_0_indexed is the integer label array currently in RAM.

print(f"X shape for splitting: {X.shape}")
print(f"y (label) shape for splitting: {y_int_0_indexed.shape}")

# Reserve validation set from the remaining data (80%) (e.g., 15% of total)
# Calculation: 0.15 / (1.0 - 0.20) = 0.1875
VAL_RATE_ADJUSTED = VAL_SIZE_RATE / (1.0 - TEST_SIZE_RATE)

# 1. Split: Train+Val and Test sets
# Using 1D integer label array (y_int_0_indexed) for stratification.
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y_int_0_indexed, # Corrected: Using y_int_0_indexed instead of y.
    test_size=TEST_SIZE_RATE,
    random_state=42,
    stratify=y_int_0_indexed # Use 1D integer labels for stratification.
)

# 2. Split: Train and Validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=VAL_RATE_ADJUSTED,
    random_state=42,
    stratify=y_train_val # Use 1D integer labels for stratification.
)

print("\n--- Data Splitting Results ---")
print(f"TRAIN Set: {len(X_train)} images (Shape: {X_train.shape})")
print(f"VAL Set (Validation): {len(X_val)} images (Shape: {X_val.shape})")
print(f"TEST Set (Final Test): {len(X_test)} images (Shape: {X_test.shape})")

print(f"\nTraining set label (y_train) shape: {y_train.shape}")
print(f"Test set label (y_test) shape: {y_test.shape}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import random
import math

# NOTE: X (images) and y_labels (string labels) must be loaded from previous cells.
# We will use the variable Y_INT_0_INDEXED (integer labels) if needed, but primarily y_labels here.

# Assuming X and y_labels are loaded
try:
    # Get final shapes of data (From previous successful loading)
    print(f"Visualization will be performed on {X.shape[0]} filtered images.")
except NameError:
    print("ERROR: X or y_labels arrays were not loaded in previous steps. Visualization cannot proceed.")
    raise

# --- 1. Calculate Class Distribution ---
# y_labels are the string labels remaining after filtering.
y_labels_list = y_labels.tolist()
df_viz = pd.DataFrame({'class_label': y_labels_list})

class_distribution = df_viz['class_label'].value_counts().sort_index()
TOTAL_CLASSES_FOUND = len(class_distribution)

print(f"\nProcess complete. Number of classes remaining after filtering: {TOTAL_CLASSES_FOUND}")

# --- 2. Visualize Class Distribution (Bar Plot) ---
plt.figure(figsize=(18, 6))
class_distribution.plot(kind='bar', color='darkgreen')
plt.title(f'Traffic Sign Class Distribution (Remaining Classes: {TOTAL_CLASSES_FOUND})', fontsize=16)
plt.xlabel('Class ID', fontsize=12)
plt.ylabel('Number of Samples (Log Scale)', fontsize=12)
plt.yscale('log')  # Logarithmic scale to better visualize imbalanced data
plt.xticks(rotation=60, ha='right', fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# --- 3. Visualize Sample Images (Image Grid) ---

# unique_classes list contains all class IDs to be visualized (as Strings).
unique_classes = class_distribution.index.tolist()
num_samples = len(unique_classes)
random.seed(42)
random.shuffle(unique_classes)  # Shuffle classes

print(f"Visualizing each of the {TOTAL_CLASSES_FOUND} unique classes detected.")

# Set Grid Dimensions (e.g., 7 Columns)
cols = 7
rows = math.ceil(num_samples / cols)

plt.figure(figsize=(18, rows * 2.5))  # Set figure size

# Loop to show one example from each class
for i, class_id in enumerate(unique_classes):

    # Find the index of the first image belonging to that class
    try:
        # np.where finds locations where class_id (string) occurs in y_labels
        sample_index = np.where(y_labels == class_id)[0][0]
        sample_image = X[sample_index]
    except IndexError:
        # Skip if no image exists for this class (Assumes labeling is correct)
        continue

    plt.subplot(rows, cols, i + 1)

    # Image is in [0, 1] range, so it displays directly
    plt.imshow(sample_image)

    # Show Class ID and total count in the title
    count = class_distribution[class_id]
    plt.title(f'ID: {class_id} (N={count})', fontsize=10)
    plt.axis('off')

plt.suptitle(f'Samples from All {TOTAL_CLASSES_FOUND} Different Traffic Sign Classes', fontsize=18, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import random
import math

# (Assumes X and y_labels/y_int_0_indexed arrays are loaded from previous steps)

# --- 1. Calculating Class Distribution (Consistent Data Usage) ---
# Since the X array was reduced during loading and filtering steps,
# the y_labels array must also be filtered. (Proceeding with this assumption)
class_distribution = pd.Series(y_labels).value_counts().sort_index()
TOTAL_CLASSES_FOUND = len(class_distribution)

# unique_classes list contains all class IDs to be visualized.
unique_classes = class_distribution.index.tolist()
num_samples = len(unique_classes)

print(f"Visualizing each of the {TOTAL_CLASSES_FOUND} unique classes detected.")

# --- 2. Visualize Class Distribution (Bar Plot) ---
plt.figure(figsize=(18, 6))
class_distribution.plot(kind='bar', color='darkgreen')
plt.title(f'Traffic Sign Class Distribution (Remaining Classes: {TOTAL_CLASSES_FOUND})', fontsize=16)
plt.xlabel('Class ID', fontsize=12)
plt.ylabel('Number of Samples (Log Scale)', fontsize=12)
plt.yscale('log')
plt.xticks(rotation=60, ha='right', fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# --- 3. Visualize Sample Images (Image Grid) ---

random.seed(42)
random.shuffle(unique_classes)  # Shuffle the order of classes appearing in the grid

cols = 7
rows = math.ceil(num_samples / cols)

plt.figure(figsize=(18, rows * 2.5))

for i, class_id in enumerate(unique_classes):
    # Find indices of all images belonging to that class
    indices = np.where(y_labels == class_id)[0]

    try:
        # Select a random image index
        sample_index = random.choice(indices)
        sample_image = X[sample_index]
    except IndexError:
        # Skip if no image exists for this class
        continue

    plt.subplot(rows, cols, i + 1)

    # Image is in [0, 1] range, so it displays directly
    plt.imshow(sample_image)

    # Show Class ID and total count in the title
    count = class_distribution[class_id]
    plt.title(f'ID: {class_id} (N={count})', fontsize=10)
    plt.axis('off')

plt.suptitle(f'Random Samples from All {TOTAL_CLASSES_FOUND} Different Traffic Sign Classes', fontsize=18, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

print("Preparing Random Forest model...")

# STEP 1: Flattening the Data
# This step was already performed in the Decision Tree cell (Cell 10),
# but let's flatten the data from X_train and X_test again so this cell can run independently.
# (Using X_train and X_test from previous cells)

print(f"X_train shape for flattening: {X_train.shape}")
print(f"X_test shape for flattening: {X_test.shape}")

nsamples_train, nx, ny, nrgb = X_train.shape
X_train_flat = X_train.reshape((nsamples_train, nx*ny*nrgb))

nsamples_test, nx, ny, nrgb = X_test.shape
X_test_flat = X_test.reshape((nsamples_test, nx*ny*nrgb))

print(f"Training data flattened. New shape: {X_train_flat.shape}")
print(f"Test data flattened. New shape: {X_test_flat.shape}")


# STEP 2: Creating and Training the Model
# y_train and y_test are already suitable as they are 1D integer arrays.
print("Creating model...")

# n_estimators=100 -> Use 100 decision trees
# n_jobs=-1 -> Use all available processors in Colab to speed up training
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

print("Training model (This process may take a bit longer than Decision Tree)...")
rf_model.fit(X_train_flat, y_train)

print("Training complete.")

# STEP 3: Making Predictions on Test Data
print("Making predictions on test data...")
# Adding _rf suffix to variable names to avoid confusion with Decision Tree variables
y_pred_rf = rf_model.predict(X_test_flat)

# STEP 4: Evaluating Results
accuracy_rf = accuracy_score(y_test, y_pred_rf)
print("\n--- Random Forest Model Results ---")
print(f"Accuracy: {accuracy_rf * 100:.2f}%")

# More detailed report
unique_labels_in_test = np.unique(y_test)
target_names = [f"Class {i}" for i in unique_labels_in_test]

print("\nClassification Report (Classes in Test Set):")
print(classification_report(y_test, y_pred_rf, labels=unique_labels_in_test, target_names=target_names, zero_division=0))

In [ ]:
import joblib
from pathlib import Path

# Saving the model
MODEL_FILE_PATH = 'random_forest_traffic_sign.joblib'

# The rf_model variable is assumed to be created from the previous Random Forest training.
joblib.dump(rf_model, MODEL_FILE_PATH)
print(f"Random Forest model successfully saved to '{MODEL_FILE_PATH}'.")

# --- Loading on UI Side (For Verification/Simulation) ---
# loaded_model = joblib.load(MODEL_FILE_PATH)
# print("Model prepared and loaded for UI (simulation).")

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

print("Training K-Nearest Neighbors (KNN) model... (This may take some time depending on dataset size)")

# STEP 1: Defining the KNN Model
# n_neighbors (K value) is usually chosen as 3, 5, or 7.
knn_model = KNeighborsClassifier(n_neighbors=5)

# STEP 2: Training the Model
# Note: X_train_flat and y_train variables must be defined in previous cells.
knn_model.fit(X_train_flat, y_train)

# STEP 3: Prediction and Evaluation
y_pred_knn = knn_model.predict(X_test_flat)
accuracy_knn = accuracy_score(y_test, y_pred_knn)

print(f"\n--- KNN Model Results ---")
print(f"Accuracy: {accuracy_knn * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_knn))

# STEP 4: Saving the Model (If needed for UI/Test phase)
MODEL_KNN_PATH = 'knn_traffic_sign.joblib'
joblib.dump(knn_model, MODEL_KNN_PATH)
print(f"\nKNN model successfully saved as '{MODEL_KNN_PATH}'.")

In [ ]:
# Mapping from 0-Indexed Class IDs to Class Names
CLASS_NAMES = {
    0: "Speed limit (20km/h)",
    1: "Speed limit (30km/h)",
    2: "Speed limit (50km/h)",
    3: "Speed limit (60km/h)",
    4: "Speed limit (70km/h)",
    5: "Speed limit (80km/h)",
    6: "End of speed limit (80km/h)",
    7: "Speed limit (100km/h)",
    8: "Speed limit (120km/h)",
    9: "No passing",
    10: "No passing for vehicles over 3.5 metric tons",
    11: "Right-of-way at the next intersection",
    12: "Priority road",
    13: "Yield",
    14: "Stop",
    15: "No vehicles",
    16: "Vehicles over 3.5 metric tons prohibited",
    17: "No entry",
    18: "General caution",
    19: "Dangerous curve to the left",
    20: "Dangerous curve to the right",
    21: "Double curve",
    22: "Bumpy road",
    23: "Slippery road",
    24: "Road narrows on the right",
    25: "Road work",
    26: "Traffic signals",
    27: "Pedestrians",
    28: "Children crossing",
    29: "Bicycles crossing",
    30: "Beware of ice/snow",
    31: "Wild animals crossing",
    32: "End of all speed and passing limits",
    33: "Turn right ahead",
    34: "Turn left ahead",
    35: "Ahead only",
    36: "Go straight or right",
    37: "Go straight or left",
    38: "Keep right",
    39: "Keep left",
    40: "Roundabout mandatory",
    41: "End of no passing",
    42: "End of no passing by vehicles over 3.5 metric tons"
}

# For verification purposes:
# print(f"Class 17: {CLASS_NAMES[17]}")
# print(f"Class 42: {CLASS_NAMES[42]}")

In [ ]:
import joblib
from PIL import Image
import numpy as np
from pathlib import Path
import os
import math

# --- 1. LOADING CONSTANTS AND MODEL ---

# Path where the model is saved
MODEL_FILE_PATH = 'random_forest_traffic_sign.joblib'

# Target size used in training (64x64x3)
TARGET_SIZE = (64, 64)

# Mapping from 0-Indexed Class IDs to Class Names (Taken from previous step)
CLASS_NAMES = {
    0: "Speed limit (20km/h)",
    1: "Speed limit (30km/h)",
    2: "Speed limit (50km/h)",
    3: "Speed limit (60km/h)",
    4: "Speed limit (70km/h)",
    5: "Speed limit (80km/h)",
    6: "End of speed limit (80km/h)",
    7: "Speed limit (100km/h)",
    8: "Speed limit (120km/h)",
    9: "No passing",
    10: "No passing for vehicles over 3.5 metric tons",
    11: "Right-of-way at the next intersection",
    12: "Priority road",
    13: "Yield",
    14: "Stop",
    15: "No vehicles",
    16: "Vehicles over 3.5 metric tons prohibited",
    17: "No entry",
    18: "General caution",
    19: "Dangerous curve to the left",
    20: "Dangerous curve to the right",
    21: "Double curve",
    22: "Bumpy road",
    23: "Slippery road",
    24: "Road narrows on the right",
    25: "Road work",
    26: "Traffic signals",
    27: "Pedestrians",
    28: "Children crossing",
    29: "Bicycles crossing",
    30: "Beware of ice/snow",
    31: "Wild animals crossing",
    32: "End of all speed and passing limits",
    33: "Turn right ahead",
    34: "Turn left ahead",
    35: "Ahead only",
    36: "Go straight or right",
    37: "Go straight or left",
    38: "Keep right",
    39: "Keep left",
    40: "Roundabout mandatory",
    41: "End of no passing",
    42: "End of no passing by vehicles over 3.5 metric tons"
}

# Load the model
try:
    rf_model = joblib.load(MODEL_FILE_PATH)
    print(f"Random Forest Model successfully loaded from '{MODEL_FILE_PATH}'.")
except FileNotFoundError:
    print(f"ERROR: File '{MODEL_FILE_PATH}' not found. Please save the model first.")
    raise


# --- 2. PREDICTION FUNCTION ---

def preprocess_and_predict(image_path_or_array, loaded_model, class_names):
    """
    Preprocesses the given image path or array and makes a prediction using the Random Forest model.

    Args:
        image_path_or_array (str/PIL.Image/np.ndarray): Image path or PIL/NumPy image.
        loaded_model (RandomForestClassifier): Loaded RF model.
        class_names (dict): Dictionary mapping class IDs to names.

    Returns:
        dict: Prediction result (ID, Name, Confidence).
    """

    # 1. Load Image and Convert to RGB
    if isinstance(image_path_or_array, str) or isinstance(image_path_or_array, Path):
        # Load from file path
        img = Image.open(image_path_or_array).convert('RGB')
    elif isinstance(image_path_or_array, np.ndarray):
        # Load from NumPy array (convert to 0-255 range if necessary)
        img_temp = (image_path_or_array * 255).astype(np.uint8) if np.max(image_path_or_array) <= 1.0 else image_path_or_array
        img = Image.fromarray(img_temp).convert('RGB')
    else:
        # Assume it is a PIL Image object
        img = image_path_or_array.convert('RGB')

    # 2. Resize (Target size used in training)
    # Important: Use Image.Resampling.LANCZOS as done in training
    img_resized = img.resize(TARGET_SIZE, Image.Resampling.LANCZOS)

    # 3. Convert to NumPy Array and Normalize (0-1)
    img_array = np.array(img_resized).astype(np.float32) / 255.0

    # 4. Flatten and Reshape
    # Shape must be (1, 12288) derived from (64, 64, 3)
    input_vector = img_array.flatten().reshape(1, -1)

    # 5. Make Prediction
    prediction_id = loaded_model.predict(input_vector)[0]

    # Get probabilities (To show confidence level)
    prediction_proba = loaded_model.predict_proba(input_vector)[0]
    confidence = np.max(prediction_proba)

    # 6. Map to Class Name
    result_name = class_names.get(prediction_id, "Unknown Class")

    return {
        "id": prediction_id,
        "name": result_name,
        "confidence": confidence
    }

# # --- EXAMPLE USAGE ---
# # Enter the path of a new test image here:
# # new_image_path = "path/to/your/new_traffic_sign_image.png"

# # try:
# #     prediction_result = preprocess_and_predict(new_image_path, rf_model, CLASS_NAMES)

# #     print("\n--- PREDICTION RESULT ---")
# #     print(f"Predicted Class ID: {prediction_result['id']}")
# #     print(f"Predicted Class Name: {prediction_result['name']}")
# #     print(f"Confidence Level: {prediction_result['confidence']:.4f}")
# # except FileNotFoundError:
# #     print(f"ERROR: Sample image '{new_image_path}' not found.")

In [ ]:
import os
import random
import numpy as np
import joblib
from PIL import Image
from pathlib import Path
from tqdm import tqdm
from IPython.display import display, HTML

# --- 1. SETTINGS ---
MODEL_FILE_PATH = 'random_forest_traffic_sign.joblib'
IMAGE_FOLDER = 'png_images'  # Main folder where images are extracted
TARGET_SIZE = (64, 64)
TEST_SAMPLE_SIZE = 100  # Number of images you want to test

# Mapping from Class IDs to Names (Standard GTSRB English Names)
CLASS_NAMES = {
    0: "Speed limit (20km/h)", 1: "Speed limit (30km/h)", 2: "Speed limit (50km/h)",
    3: "Speed limit (60km/h)", 4: "Speed limit (70km/h)", 5: "Speed limit (80km/h)",
    6: "End of speed limit (80km/h)", 7: "Speed limit (100km/h)", 8: "Speed limit (120km/h)",
    9: "No passing", 10: "No passing for vehicles over 3.5 metric tons",
    11: "Right-of-way at the next intersection", 12: "Priority road", 13: "Yield", 14: "Stop",
    15: "No vehicles", 16: "Vehicles over 3.5 metric tons prohibited", 17: "No entry",
    18: "General caution", 19: "Dangerous curve to the left", 20: "Dangerous curve to the right",
    21: "Double curve", 22: "Bumpy road", 23: "Slippery road", 24: "Road narrows on the right",
    25: "Road work", 26: "Traffic signals", 27: "Pedestrians",
    28: "Children crossing", 29: "Bicycles crossing", 30: "Beware of ice/snow",
    31: "Wild animals crossing", 32: "End of all speed and passing limits", 33: "Turn right ahead",
    34: "Turn left ahead", 35: "Ahead only", 36: "Go straight or right",
    37: "Go straight or left", 38: "Keep right", 39: "Keep left",
    40: "Roundabout mandatory", 41: "End of no passing",
    42: "End of no passing by vehicles over 3.5 metric tons"
}

# --- 2. LOAD MODEL ---
if not os.path.exists(MODEL_FILE_PATH):
    print(f"ERROR: '{MODEL_FILE_PATH}' not found. Please run the model training cell first.")
else:
    model = joblib.load(MODEL_FILE_PATH)
    print("Model successfully loaded. Starting testing process...\n")

    # --- 3. SELECT RANDOM TEST IMAGES ---
    all_images = [str(p) for p in Path(IMAGE_FOLDER).glob("*.png")]

    if len(all_images) == 0:
        print(f"ERROR: No photos found in folder '{IMAGE_FOLDER}'.")
    else:
        # If fewer photos than specified, take all; otherwise select randomly
        sample_count = min(TEST_SAMPLE_SIZE, len(all_images))
        test_images = random.sample(all_images, sample_count)

        correct_predictions = 0
        test_results = []

        # --- 4. PREDICTION LOOP ---
        for img_path in tqdm(test_images, desc="Predicting"):
            try:
                # Get True Label from Filename (e.g., '5' from '5_123.png')
                true_id = int(Path(img_path).stem.split('_')[0])

                # Prepare Image (Same processing as training)
                img = Image.open(img_path).convert('RGB')
                img = img.resize(TARGET_SIZE)
                img_array = np.array(img) / 255.0  # Normalization
                img_flatten = img_array.reshape(1, -1)  # Vectorize for Random Forest

                # Make Prediction
                prediction_id = model.predict(img_flatten)[0]

                # Confidence score (If probability estimation exists)
                if hasattr(model, "predict_proba"):
                    confidence = np.max(model.predict_proba(img_flatten))
                else:
                    confidence = 1.0

                is_correct = (prediction_id == true_id)
                if is_correct:
                    correct_predictions += 1

                test_results.append({
                    'file_name': os.path.basename(img_path),
                    'true_id': true_id,
                    'true_name': CLASS_NAMES.get(true_id, f"ID: {true_id}"),
                    'prediction_id': prediction_id,
                    'prediction_name': CLASS_NAMES.get(prediction_id, f"ID: {prediction_id}"),
                    'is_correct': is_correct,
                    'confidence': confidence
                })
            except Exception as e:
                print(f"Error: Problem processing {img_path}: {e}")

        # --- 5. PERFORMANCE REPORT ---
        accuracy = (correct_predictions / sample_count) * 100

        print(f"\n" + "="*30)
        print(f"PERFORMANCE RESULTS")
        print(f"="*30)
        print(f"Total Tested: {sample_count}")
        print(f"Correct Predictions: {correct_predictions}")
        print(f"Wrong Predictions: {sample_count - correct_predictions}")
        print(f"TEST ACCURACY: {accuracy:.2f}%")
        print("="*30)

        # Detailed Results Table (HTML)
        table_html = """
        <h3 style='color: #2c3e50;'>Detailed Prediction List (First 50 Results)</h3>
        <table border='1' style='border-collapse: collapse; width: 100%; font-family: sans-serif;'>
            <tr style='background-color: #34495e; color: white;'>
                <th style='padding: 8px;'>File</th>
                <th style='padding: 8px;'>True Class</th>
                <th style='padding: 8px;'>Predicted</th>
                <th style='padding: 8px;'>Result</th>
                <th style='padding: 8px;'>Confidence</th>
            </tr>
        """

        for res in test_results[:50]:  # Show only the first 50 results to avoid a long table
            color = "#27ae60" if res['is_correct'] else "#e74c3c"
            status = "✓ CORRECT" if res['is_correct'] else "✗ WRONG"

            table_html += f"""
            <tr>
                <td style='padding: 5px;'>{res['file_name']}</td>
                <td style='padding: 5px;'>{res['true_name']}</td>
                <td style='padding: 5px;'>{res['prediction_name']}</td>
                <td style='padding: 5px; color: {color}; font-weight: bold;'>{status}</td>
                <td style='padding: 5px;'>{res['confidence']*100:.1f}%</td>
            </tr>
            """
        table_html += "</table>"

        if sample_count > 50:
            table_html += f"<p><i>* Only the first 50 results are shown as the list is too long.</i></p>"

        display(HTML(table_html))